# Phishing Website Detection — Experimental Analysis

**Data source:** [UCI Phishing Websites, dataset 327](https://archive.ics.uci.edu/dataset/327/phishing+websites) — 11,055 real labeled website observations (30 numeric features).

The JSONL rows here are **new local measurements** of model inference on UCI observations, not pre-existing HTTP server access logs. Training and warm-up occur before benchmark statistics.

**Reproduction:** Run `python -m src.download_uci`, then `python -m src.collect_inference --data data/external/uci327_phishing_websites.csv --runs 120 --out data/raw/test_run.jsonl` from repository root. Then use *Restart & Run All*.

Do not submit previous 10-row smoke-test output or constant-latency dump as real measurements. Notebook outputs are intentionally cleared until actual UCI logs exist.


# Reproducible ML Inference Benchmark

**Goal:** process raw JSONL measurements, exclude cold-start warm-up runs, calculate error rates and seeded bootstrap uncertainty, and export publication-style figures. **Dataset:** UCI 327 (11,055 real labeled website records). Results are generated only after running the real measurement collector on Linux.

## 1. Environment and reproducibility
Run **Kernel → Restart & Run All**. All paths resolve from the project root. Randomness in Python and NumPy is fixed to **42**; bootstrap sampling uses a separately seeded local generator inside `calculate_statistics`. CSV and graphics are written with deterministic filenames.

In [1]:
from pathlib import Path
import sys, random
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.process_data import read_telemetry, load_and_clean_data, calculate_statistics
from src.visualize import generate_plots
SEED=42
random.seed(SEED)
np.random.seed(SEED)
print("Project:", ROOT, "| Seed:", SEED)

Project: /home/darlene/Documents/phishing-ml-benchmark-real-data-linux | Seed: 42


## 2. Data ingestion and schema validation
Each JSONL record must supply `phase`, `model_variant`, `batch_size`, `latency_ms`, and `status_code`. `batch_size` is **not** requests-per-second. The locally measured `test_run.jsonl` is excluded from Git; it must be generated with the UCI dataset before executing this notebook.

In [2]:
RAW = ROOT / "data/raw/test_run.jsonl"
raw = read_telemetry(RAW)
print("Total observations:", len(raw))
print("Phase counts:")
display(raw.groupby("phase").size().rename("records").to_frame())
print("Model / batch counts:")
display(raw.groupby(["model_variant","batch_size","phase"]).size().unstack(fill_value=0))

Total observations: 1200
Phase counts:


,records
phase,
benchmark,1080
warmup,120


Model / batch counts:


phase                          benchmark  warmup
model_variant      batch_size                   
LogisticRegression 1                 108      12
                   2                 108      12
                   4                 108      12
                   6                 108      12
                   9                 108      12
RandomForest       1                 108      12
                   2                 108      12
                   4                 108      12
                   6                 108      12
                   9                 108      12

## 3. Warm-up filtering and failure handling
Warm-up measurements have cold-cache/start-up bias and must be excluded from benchmark statistics. **Failures must not disappear:** they remain in each group’s error-rate denominator; only valid successful calls contribute to latency percentiles.

In [3]:
benchmark = load_and_clean_data(RAW)
n_warmup=int((raw.phase=="warmup").sum())
assert n_warmup > 0, "Fixture must exercise warm-up removal"
assert len(benchmark)+n_warmup == len(raw)
assert benchmark.phase.eq("benchmark").all()
print(f"Excluded warm-up: {n_warmup}; retained benchmark rows: {len(benchmark)}")
print(f"Benchmark failures: {(benchmark.status_code!=200).sum()}")

Excluded warm-up: 120; retained benchmark rows: 1080
Benchmark failures: 0


## 4. Descriptive statistics + 95% bootstrap CI
For each model and batch size: calculate sample count, failed calls, error rate (%), mean ± SD, median, p25/p75, p95 and p99. Bootstrap **2,000** re-sampled medians with replacement. Use the 2.5th and 97.5th percentiles as a percentile 95% CI, with seed **42**. Short sequential timings may be correlated, so this is an illustrative uncertainty interval.

In [4]:
summary=calculate_statistics(benchmark,seed=SEED,n_boot=2000)
assert (summary.n_total==summary.n_success+summary.n_error).all()
assert (summary.median_ci_lower_ms<=summary.median_ms).all()
assert (summary.median_ms<=summary.median_ci_upper_ms).all()
display(summary.round(4))
CSV=ROOT/"data/processed/metrics_summary.csv"
CSV.parent.mkdir(parents=True,exist_ok=True)
summary.to_csv(CSV,index=False)
print("Saved:",CSV)

,model_variant,batch_size,n_total,n_success,n_error,error_rate_pct,mean_ms,sd_ms,median_ms,p25_ms,p75_ms,p95_ms,p99_ms,median_ci_lower_ms,median_ci_upper_ms
0,LogisticRegression,1,108,108,0,0.0,2.1041,0.2081,2.1868,2.1370,2.2213,2.2803,2.3055,2.1725,2.1985
1,LogisticRegression,2,108,108,0,0.0,1.7407,0.0619,1.7363,1.6977,1.7691,1.8375,1.9052,1.7214,1.7492
2,LogisticRegression,4,108,108,0,0.0,1.7115,0.0552,1.7083,1.6833,1.7371,1.7961,1.8687,1.7020,1.7193
3,LogisticRegression,6,108,108,0,0.0,1.7183,0.0612,1.7197,1.6947,1.7460,1.8125,1.8568,1.7065,1.7281
4,LogisticRegression,9,108,108,0,0.0,1.7023,0.0809,1.7060,1.6488,1.7548,1.8357,1.8820,1.6891,1.7265
5,RandomForest,1,108,108,0,0.0,14.2284,0.5406,13.9797,13.8879,14.3208,15.3766,15.6353,13.9392,14.0453
6,RandomForest,2,108,108,0,0.0,14.8466,0.7515,14.9742,14.0973,15.2683,16.2726,16.6530,14.6566,15.0825
7,RandomForest,4,108,108,0,0.0,14.4034,0.5119,14.1752,14.0341,14.6635,15.4695,15.7631,14.1387,14.2384
8,RandomForest,6,108,108,0,0.0,14.5489,0.6198,14.2968,14.1602,14.9348,15.5040,15.8172,14.2136,14.3888
9,RandomForest,9,108,108,0,0.0,15.1418,1.0695,14.7020,14.4514,15.6566,16.4414,18.2625,14.5661,15.1621


Saved: /home/darlene/Documents/phishing-ml-benchmark-real-data-linux/data/processed/metrics_summary.csv


## 5. Scientific visualization
**Figure 1:** median latency (ms) vs batch size, with **95% bootstrap CI error bars**, distinct markers and line styles. **Figure 2:** boxplot of latency distributions for the two model variants (pooled across all batch sizes; warm-up excluded). Axis units are explicit. Both plots are saved as **vector PDF and 300 DPI PNG**.

In [5]:
FIG=ROOT/"reports/figures"
generate_plots(benchmark,FIG,seed=SEED)
files=sorted(FIG.glob("fig*"))
for f in files:print(f.relative_to(ROOT),f.stat().st_size,"bytes")
assert all((FIG/f"{name}.{ext}").is_file() for name in ["fig1_latency_vs_load","fig2_distribution_boxplot"] for ext in ["pdf","png"])

reports/figures/fig1_latency_vs_load.pdf 18183 bytes
reports/figures/fig1_latency_vs_load.png 132358 bytes
reports/figures/fig2_distribution_boxplot.pdf 15905 bytes
reports/figures/fig2_distribution_boxplot.png 116925 bytes


## 6. Results & Discussion (limitations)
This notebook analyzes **actual local inference measurements on UCI 327**, once generated. It does not establish which model is superior in phishing classification: this benchmark measures the predict latency only, not held-out F1 or recall. Batch size is not production RPS; timings depend on Linux hardware, background noise, and CPU scheduling. Report model quality separately and avoid claiming production traffic traces.parately from inference timing. See `docs/domain_model.md` for mathematical definitions.